# Estudio de penetración indoor en edificio real

Objetivo: evaluar si la cobertura macro LTE/UMTS exterior es suficiente para voz, mensajería y datos básicos en 3 puntos interiores y proponer refuerzo (DAS/repetidor/small cell).

Herramientas: Python + Jupyter + QGIS (opcional).

## 1) Parámetros de referencia

- Frecuencias: LTE 1800 MHz y UMTS 2100 MHz (backup).
- Antena macro exterior: 20-25 m de altura.
- Potencia Tx macro: 43 dBm.
- Ganancias: Gtx 15 dBi, Grx 0 dBi.
- Pérdidas de penetración típicas: fachada 12-18 dB, tabique 3-5 dB, forjado 12-18 dB.
- Umbrales de servicio: voz -95 dBm, datos básicos -85 dBm.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Ptx = 43  # dBm macro Tx
Gtx = 15
Grx = 0

# Umbrales de servicio
umbral_voz = -95
umbral_datos = -85

# Puntos de usuario
points = [
    {'punto': 'P1 entrada', 'dist_m': 80, 'muro_fachada': 15, 'muro_interior': 3, 'forjado': 0, 'nivel_servicio': 'entrada'},
    {'punto': 'P2 aula 2ª', 'dist_m': 120, 'muro_fachada': 15, 'muro_interior': 6, 'forjado': 15, 'nivel_servicio': 'aula'},
    {'punto': 'P3 semisótano', 'dist_m': 100, 'muro_fachada': 15, 'muro_interior': 3, 'forjado': 18, 'nivel_servicio': 'semi'}
]

def pl_cost231(d_km, f_mhz, h_bts=25, h_mt=1.5):
    # COST231-Hata (en dB) para zona urbana d>=1km; ajustamos con d en km
    a_hm = (1.1*np.log10(f_mhz)-0.7)*h_mt - (1.56*np.log10(f_mhz)-0.8)
    pl = 46.3 + 33.9*np.log10(f_mhz) - 13.82*np.log10(h_bts) - a_hm + (44.9 - 6.55*np.log10(h_bts))*np.log10(d_km) + 3
    return pl

def calc_result(p, f_mhz=1800):
    d_km = max(0.001, p['dist_m']/1000)
    pl_ext = pl_cost231(d_km, f_mhz)
    pl_pen = p['muro_fachada'] + p['muro_interior'] + p['forjado']
    pl_tot = pl_ext + pl_pen
    prx = Ptx + Gtx + Grx - pl_tot
    return pl_ext, pl_pen, pl_tot, prx

rows = []
for p in points:
    pl_ext, pl_pen, pl_tot, prx = calc_result(p, f_mhz=1800)
    rows.append({
        'punto': p['punto'],
        'dist_m': p['dist_m'],
        'PL_ext_1800': round(pl_ext,1),
        'PL_pen': round(pl_pen,1),
        'PL_tot_1800': round(pl_tot,1),
        'Prx_1800': round(prx,1),
        'cumple_voz': 'SI' if prx >= umbral_voz else 'NO',
        'cumple_datos': 'SI' if prx >= umbral_datos else 'NO'
    })
df = pd.DataFrame(rows)
df

# Comparación rápida de frecuencias 700/1800/2600
freqs = [700, 1800, 2600]
res = []
for p in points:
    for f in freqs:
        pl_ext, pl_pen, pl_tot, prx = calc_result(p, f_mhz=f)
        res.append({'punto': p['punto'], 'f_mhz': f, 'Prx': round(prx,1), 'PL_tot': round(pl_tot,1)})
res_df = pd.DataFrame(res)
res_df

fig, ax = plt.subplots(figsize=(8,4))
for p in points:
    subset = res_df[res_df.punto == p['punto']]
    ax.plot(subset.f_mhz, subset.Prx, marker='o', label=p['punto'])
ax.axhline(umbral_voz, color='k', linestyle='--', label='Umbral voz (-95)')
ax.axhline(umbral_datos, color='r', linestyle='--', label='Umbral datos (-85)')
ax.set_xlabel('Frecuencia (MHz)')
ax.set_ylabel('Prx estimado (dBm)')
ax.set_title('Comparación de Prx en frecuencias 700/1800/2600')
ax.legend()
ax.grid(True)
plt.show()

## 2) Interpretación y recomendaciones

- Si `P3 semisótano` o `P2 aula` no cumplen voz/datos con 1800, el refuerzo es necesario.
- Propuesta de refuerzo: DAS con antenas en semisótano y planta 2, o small cell 1800/2100 dentro de zonas críticas.
- Alternativa: usar 700/800 para mejores ganancias de penetración fachada (especialmente semisótano).
- Repetidor solo si hay línea de vista macro y no se desea instalación de DAS, poniendo boato en 1-2 puntos y comprobando retroalimentación.